# QQQI / SGOV blended-defense episode attribution

This notebook explains the drawdown-depth versus recovery-duration trade-off of the frozen blended defensive profile. It does not modify v4.2 signals, thresholds, weights, execution timing, or the 10 bps cost convention.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('artifacts/evidence/qqqi_qqq_tqqq_v4_2_sgov_episode_attribution')
episodes = pd.read_csv(ROOT / 'drawdown_episode_attribution.csv', parse_dates=['episode_start', 'baseline_trough_date', 'baseline_recovery_date', 'challenger_trough_date', 'challenger_recovery_date'])
major = pd.read_csv(ROOT / 'major_drawdown_episodes.csv', parse_dates=['episode_start', 'baseline_trough_date', 'baseline_recovery_date', 'challenger_trough_date', 'challenger_recovery_date'])
phase_state = pd.read_csv(ROOT / 'phase_state_log_relative_contribution.csv')
headline = pd.read_csv(ROOT / 'headline_metrics.csv')
summary = json.loads((ROOT / 'experiment_summary.json').read_text(encoding='utf-8'))


## Headline context and monitor-only gate

In [ ]:
display(headline[['strategy', 'cagr', 'annual_volatility', 'sharpe', 'sortino', 'max_drawdown', 'calmar']])
pd.DataFrame([summary['prospective_monitor_gate']['metrics']]).T.rename(columns={0: 'value'})


## Five largest v4.2 drawdown episodes

In [ ]:
columns = [
    'episode_id', 'episode_start', 'baseline_trough_date',
    'baseline_max_drawdown', 'challenger_max_drawdown',
    'drawdown_improvement_pp', 'baseline_recovery_sessions',
    'challenger_recovery_sessions', 'recovery_lag_sessions',
    'chronological_segment',
]
display(major.sort_values('severity_rank')[columns])


## Drawdown benefit versus recovery lag

In [ ]:
plot_data = major.dropna(subset=['recovery_lag_sessions']).copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_data['recovery_lag_sessions'], plot_data['drawdown_improvement_pp'])
for _, row in plot_data.iterrows():
    ax.annotate(row['episode_id'], (row['recovery_lag_sessions'], row['drawdown_improvement_pp']))
ax.axhline(0.0, linewidth=1)
ax.axvline(30.0, linewidth=1, linestyle='--')
ax.set_xlabel('Challenger recovery lag (sessions)')
ax.set_ylabel('Drawdown improvement (percentage points)')
ax.set_title('Major episode trade-off')
plt.show()


## Exact log-relative-wealth attribution by phase and state

In [ ]:
major_phase = phase_state.loc[phase_state['scope'].eq('major_episodes')]
display(major_phase.pivot(index='phase', columns='position_state', values='total_log_relative'))
summary['prospective_monitor_gate']


## Interpretation rule

A shallower trough is not sufficient. The monitor-only gate jointly tests episode consistency, recovery lag, concentration, chronological stability, and the full-sample CAGR sacrifice. Passing authorizes only a separate research monitor; v4.2 remains the baseline.